In [ ]:
from folktexts.llm_utils import load_model_tokenizer
from folktexts import TransformersLLMClassifier
from folktexts.acs import ACSDataset

from folktexts.task import TaskMetadata
from folktexts.benchmark import Benchmark


### Load dataset

In [2]:
acs_task_name = "ACSIncome"     # Name of the benchmark ACS task to use
acs_task = TaskMetadata.get_task(acs_task_name)
dataset = ACSDataset.make_from_task(acs_task_name, cache_dir='./data')   # use `.subsample(0.01)` to get faster approximate results
X_test, y_test = dataset.get_test()
example_row = X_test.iloc[0]

Loading ACS data...


### Create task with subset of the features

In [7]:
acs_task_subset = TaskMetadata.create_task_with_feature_subset(acs_task, feature_subset=['AGEP', 'SCHL', 'OCCP', 'WKHP'],)

ERROR:root:A task with `name='ACSIncome_AGEP_OCCP_SCHL_WKHP'` already exists. Overwriting...


### Load Model

In [ ]:
# Load transformers model
model, tokenizer = load_model_tokenizer("gpt2")   # using tiny model as an example

# Create an object that classifies data using an LLM
clf = TransformersLLMClassifier(
    model=model,
    tokenizer=tokenizer,
    task=acs_task_subset,
)

print(clf._encode_row(example_row))

/Users/mgorecki/opt/miniconda3/envs/llm-py311/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


The following data corresponds to a survey respondent. The survey was conducted among US residents in 2018. Please answer the question based on the information provided. The data provided is enough to reach an approximate answer.

Information:
- age is 23 years old
- highest educational attainment is Some college, 1 or more years, no degree
- occupation is Retail salespersons
- usual number of hours worked per week is 20 hours

Question: What is this person's estimated yearly income?
A. Below $50,000.
B. Above $50,000.
Answer:


### Change subset when running becnhmark

In [16]:
subsampling_ratio = 0.01
bench = Benchmark.make_acs_benchmark(
    model= model,
    tokenizer=tokenizer,
    task_name="ACSIncome",
    subsampling=subsampling_ratio,
    numeric_risk_prompting=True,
    feature_subset = ['AGEP'],#, 'SCHL', 'OCCP', 'WKHP'],
    data_dir='./data'
)

Loading ACS data...
Using zero-shot prompting.


In [17]:
bench.run(results_root_dir='./results/test')

Computing risk estimates:   0%|          | 0/105 [00:00<?, ?it/s]

/Users/mgorecki/Documents/projects/llm_fairness/folktexts/folktexts/plotting.py:110: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(
/Users/mgorecki/Documents/projects/llm_fairness/folktexts/folktexts/plotting.py:118: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.xlim(y_pred_scores.min(), y_pred_scores.max())


{'threshold': 0.5,
 'n_samples': 1665,
 'n_positives': 645,
 'n_negatives': 1020,
 'model_name': 'gpt2',
 'accuracy': 0.38738738738738737,
 'tpr': 1.0,
 'fnr': 0.0,
 'fpr': 1.0,
 'tnr': 0.0,
 'balanced_accuracy': 0.5,
 'precision': 0.38738738738738737,
 'ppr': 1.0,
 'num_samples': 1665,
 'num_positives': 645,
 'num_negatives': 1020,
 'num_pred_positives': 1665,
 'num_pred_negatives': 0,
 'log_loss': 0.7138017972446689,
 'brier_score_loss': 0.2603047837837838,
 'fnr_ratio': 0,
 'fnr_diff': 0.0,
 'balanced_accuracy_ratio': 1.0,
 'balanced_accuracy_diff': 0.0,
 'tnr_ratio': 0,
 'tnr_diff': 0.0,
 'ppr_ratio': 1.0,
 'ppr_diff': 0.0,
 'fpr_ratio': 1.0,
 'fpr_diff': 0.0,
 'precision_ratio': 0.39612676056338025,
 'precision_diff': 0.3220657276995305,
 'tpr_ratio': 1.0,
 'tpr_diff': 0.0,
 'accuracy_ratio': 0.39612676056338025,
 'accuracy_diff': 0.3220657276995305,
 'equalized_odds_ratio': 0,
 'equalized_odds_diff': 0.0,
 'accuracy_group=1': 0.4050925925925926,
 'tpr_group=1': 1.0,
 'fnr_group=1